# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요
- 행/열: 1,500 × 8
- 주요 컬럼: order_id, customer_age, category, channel, price, quantity, amount, return_amount

## 2. 진단 결과
- 결측: amount 51건(3.40%), price 5건(0.33%).
현재 출력만으로는 결측이 특정 채널에 집중됐는지에 대한 매커니즘과 그룹 편중은 추가 분석이 필요함.
- 이상치(IQR):

display( partner.loc[ partner["customer_age"].isin([0, 999]), ["order_id", "customer_age"] ] ) display( partner.loc[ partner["quantity"] == 200, ["order_id", "quantity", "amount"] ] ) display( partner.loc[ partner["amount"] == 50_000_000, ["order_id", "category", "channel", "quantity", "amount"] ] )

customer_age(999·0, 1건), quantity(200, 1건), amount(50,000,000, 1건),
- 의심되는 결측 유형: customer_age(999·0)은 단순 입력 실수 가정, amount는 app 채널 편중 쏠림으로 MAR로 가정할 수 있으나, 오류 또는 특수 사례 여부 확인 필요.

## 3. 처리 결정과 근거
| 컬럼·이슈 | 결정 | 근거(한 줄) | 한계(한 줄) |
|---|---|---|---|
| `amount` 결측 | 채널별 중앙값 대체 | app 채널의 결측 비율이 높아 MAR 가설에 부합 | 채널 외 다른 원인은 검토하지 않음 |
| `price` 결측 | 카테고리별 중앙값 대체 | 결측 비율이 0.3%로 적고, 카테고리별 가격대가 다름 | 표본이 적어 카테고리 통계가 불안정할 수 있음 |
| `customer_age = 999, 0` | `NaN` 표시 후 중앙값 대체 | 물리적으로 불가능한 값이므로 입력 오류로 추정 | 외부 인증 데이터가 없어 실제 오류 여부를 확정할 수 없음 |
| `quantity = 200` | amount와 price를 통해 수량값을 복원하고 원본값·수정 플래그 보존 |  중앙값 대체는 해당 주문의 정보를 잃지만, 존재하는 해당 주문의 price와 amount 정보를 활요하여 수량을 직접 추정할 수 있어 정보 손실이 더 적음 | price와 amount 값이 정확하다는 전제에 의존하므로, 두 값 중 하나가 오류라면 추정 수량도 잘못될 수 있음 |
| `amount = 50,000,000` | 유지 후 `amount_outlier` 플래그 생성 | 1건의 정상적인 고액 거래일 가능성을 보존 | 도매 거래 여부를 별도로 분석해야 함 |

## 4. 처리 후 검증
- 결측 0건(설계상 NaN 유지가 필요한 컬럼 제외)
- customer_age 범위: 5 ~ 60 (정상)
- IQR 기준 amount_outlier 플래그 145건 보존(이 중 amount = 50,000,000인 극단값은 1건)

## 5. 후속 권고
- 도매 가능 고객 식별을 위해 customer_id 단위 과거 이력 확보 필요

## 필수 과제

- 결측 진단 5종 세트(열별 수·비율·행별·특정 열 결측 행·히트맵)를 모두 적용했다.

- 각 결측의 **유형(MCAR·MAR·MNAR)**을 가설로 추정하고 한 줄 근거를 적었다.

- IQR로 수치형 컬럼의 이상치를 탐지했다.

- 결측·이상치 각각에 처리 결정과 한 줄 근거·한계를 적었다.

- 처리 전후 비교 출력(결측 수, 통계량)을 노트북에 남겼다.

## 심화 과제 (선택)

- 같은 결측을 두 가지 다른 방법(예: 평균 대체 vs 채널별 중앙값 대체)으로 처리해, 결과 통계량이 어떻게 달라지는지 한 표로 비교했다.

- amount 박스플롯을 처리 전후로 두 개 그려, 클리핑/플래그 정책이 시각적으로 어떻게 다른지 확인했다.